# Chapter 17: Three Recipes — Recipe 2: The Reasoner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part4_recipes/18_recipe_reasoner.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt, 2025)  
> **Chapter 17:** Three Recipes — Recipe 2: The Reasoner  
> **Notebook:** `part4_recipes/18_recipe_reasoner.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 17 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.

**To open in Colab:** Click the badge above, or replace `github.com` with `githubtocolab.com` in the URL.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Environment setup — install only on Colab
# ---------------------------------------------------------------------------
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    %pip install -q transformers==4.40.0 trl==0.8.6 datasets==2.19.0 \
                    accelerate==0.29.3 torch matplotlib numpy

In [ ]:
# ---------------------------------------------------------------------------
# 1. Imports and reproducibility
# ---------------------------------------------------------------------------
import re, math, copy, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset as HFDataset
from collections import Counter

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## Phase 1 — Reasoning SFT Cold Start (Chain-of-Thought)

Before applying RL we teach the model the *chain-of-thought format*: each answer must spell out intermediate steps before the final number. This structured format is what the verifier (Phase 2) will check.

Format:
```
Q: What is 3*7+2?
A: Let me solve step by step.
Step 1: 3*7=21
Step 2: 21+2=23
Answer: 23
```

In [ ]:
# ---------------------------------------------------------------------------
# 2. Chain-of-thought SFT dataset — 15 arithmetic examples
# ---------------------------------------------------------------------------
COT_EXAMPLES = [
    {"q": "What is 3*7+2?",
     "a": "Let me solve step by step.\nStep 1: 3*7=21\nStep 2: 21+2=23\nAnswer: 23"},
    {"q": "What is 12-4*2?",
     "a": "Let me solve step by step.\nStep 1: 4*2=8\nStep 2: 12-8=4\nAnswer: 4"},
    {"q": "What is (5+3)*4?",
     "a": "Let me solve step by step.\nStep 1: 5+3=8\nStep 2: 8*4=32\nAnswer: 32"},
    {"q": "What is 100/5+6?",
     "a": "Let me solve step by step.\nStep 1: 100/5=20\nStep 2: 20+6=26\nAnswer: 26"},
    {"q": "What is 2**8?",
     "a": "Let me solve step by step.\nStep 1: 2**8=2*2*2*2*2*2*2*2=256\nAnswer: 256"},
    {"q": "What is 15+23-7?",
     "a": "Let me solve step by step.\nStep 1: 15+23=38\nStep 2: 38-7=31\nAnswer: 31"},
    {"q": "What is 9*9-9?",
     "a": "Let me solve step by step.\nStep 1: 9*9=81\nStep 2: 81-9=72\nAnswer: 72"},
    {"q": "What is (10+2)*(10-2)?",
     "a": "Let me solve step by step.\nStep 1: 10+2=12\nStep 2: 10-2=8\nStep 3: 12*8=96\nAnswer: 96"},
    {"q": "What is 50/2/5?",
     "a": "Let me solve step by step.\nStep 1: 50/2=25\nStep 2: 25/5=5\nAnswer: 5"},
    {"q": "What is 3**3+3?",
     "a": "Let me solve step by step.\nStep 1: 3**3=27\nStep 2: 27+3=30\nAnswer: 30"},
    {"q": "What is 7*8-14?",
     "a": "Let me solve step by step.\nStep 1: 7*8=56\nStep 2: 56-14=42\nAnswer: 42"},
    {"q": "What is (4+6)*(3+2)?",
     "a": "Let me solve step by step.\nStep 1: 4+6=10\nStep 2: 3+2=5\nStep 3: 10*5=50\nAnswer: 50"},
    {"q": "What is 144/12+8?",
     "a": "Let me solve step by step.\nStep 1: 144/12=12\nStep 2: 12+8=20\nAnswer: 20"},
    {"q": "What is 6*6+6*6?",
     "a": "Let me solve step by step.\nStep 1: 6*6=36\nStep 2: 6*6=36\nStep 3: 36+36=72\nAnswer: 72"},
    {"q": "What is 200-15*7?",
     "a": "Let me solve step by step.\nStep 1: 15*7=105\nStep 2: 200-105=95\nAnswer: 95"},
]

def format_cot(ex):
    return f"Q: {ex['q']}\nA: {ex['a']}<|endoftext|>"

cot_texts = [format_cot(e) for e in COT_EXAMPLES]
print(f'CoT SFT examples: {len(cot_texts)}')
print('Sample:')
print(cot_texts[0])

In [ ]:
# ---------------------------------------------------------------------------
# 3. Load distilgpt2 and run SFT
# ---------------------------------------------------------------------------
from trl import SFTTrainer, SFTConfig

MODEL_NAME = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

sft_dataset = HFDataset.from_dict({'text': cot_texts})

sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

sft_cfg = SFTConfig(
    output_dir='./reasoner_sft',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    save_steps=200,
    fp16=torch.cuda.is_available(),
    max_seq_length=128,
    dataset_text_field='text',
    report_to='none',
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_cfg,
    train_dataset=sft_dataset,
)

print('Training reasoning SFT ...')
result = sft_trainer.train()
print(f'SFT complete. Final loss: {result.training_loss:.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# 4. Test: can the model produce structured chain-of-thought?
# ---------------------------------------------------------------------------
def generate_cot(model, tokenizer, question, max_new_tokens=80):
    prompt = f"Q: {question}\nA: Let me solve step by step.\n"
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

TEST_QUESTIONS_SFT = [
    'What is 5*6-4?',
    'What is (8+2)*3?',
    'What is 81/9+5?',
]

print('SFT model — chain-of-thought generation examples:')
for q in TEST_QUESTIONS_SFT:
    print(f'\nQ: {q}')
    print(generate_cot(sft_model, tokenizer, q))

---
## Phase 2 — GRPO with Verifier

**Group Relative Policy Optimisation (GRPO)** generates G rollouts per prompt, computes a verifiable reward for each, normalises advantages within the group, and updates the policy with a clipped surrogate objective — no separate value network needed.

**Verifier:** we extract the predicted number after `"Answer:"` with a regex and compare it to the ground-truth.

$$A_i = \frac{r_i - \bar{r}}{\sigma_r + \epsilon}$$

$$\mathcal{L}_{GRPO} = -\mathbb{E}\left[\min\left(\rho_i A_i,\ \text{clip}(\rho_i, 1{-}\varepsilon, 1{+}\varepsilon)A_i\right)\right]$$

In [ ]:
# ---------------------------------------------------------------------------
# 5. Verifiable arithmetic dataset — 20 problems
# ---------------------------------------------------------------------------
def make_problem(expr):
    """Generate (question, ground_truth) for a safe arithmetic expression."""
    ans = int(eval(expr))  # safe: all expressions are hard-coded
    return {"question": f"What is {expr}?", "answer": ans}

ARITH_PROBLEMS = [
    make_problem("3*7+2"),
    make_problem("12-4*2"),
    make_problem("(5+3)*4"),
    make_problem("100//5+6"),
    make_problem("15+23-7"),
    make_problem("9*9-9"),
    make_problem("(10+2)*(10-2)"),
    make_problem("50//2//5"),
    make_problem("7*8-14"),
    make_problem("(4+6)*(3+2)"),
    make_problem("144//12+8"),
    make_problem("6*6+6*6"),
    make_problem("200-15*7"),
    make_problem("4*4*4"),
    make_problem("(20+4)//6"),
    make_problem("100-3*17"),
    make_problem("8*8+8"),
    make_problem("(7+3)*(7-3)"),
    make_problem("250//5-10"),
    make_problem("2+3*4+5"),
]

# Split 15 train / 5 test
grpo_train = ARITH_PROBLEMS[:15]
grpo_test  = ARITH_PROBLEMS[15:]
print(f'GRPO train: {len(grpo_train)}, test: {len(grpo_test)}')
print('Sample:', grpo_train[0])

In [ ]:
# ---------------------------------------------------------------------------
# 6. Verifier — extract predicted answer and compare to ground truth
# ---------------------------------------------------------------------------
def verify_answer(response: str, ground_truth: int) -> float:
    """
    Extract the number after 'Answer:' and compare to ground_truth.
    Returns 1.0 if correct, 0.0 if wrong or unparseable.
    """
    match = re.search(r'Answer:\s*(-?\d+)', response)
    if match is None:
        return 0.0
    predicted = int(match.group(1))
    return 1.0 if predicted == ground_truth else 0.0

# Quick sanity check
assert verify_answer("Step 1: 3*7=21\nStep 2: 21+2=23\nAnswer: 23", 23) == 1.0
assert verify_answer("Answer: 99", 23) == 0.0
assert verify_answer("No number here", 23) == 0.0
print('Verifier sanity checks passed.')

In [ ]:
# ---------------------------------------------------------------------------
# 7. Token-log-probability helper for policy gradient
# ---------------------------------------------------------------------------
def sequence_log_prob(model, tokenizer, prompt_text, response_text, max_length=128):
    """
    Compute the sum of log-probabilities of response_text tokens given prompt_text.
    """
    full = prompt_text + response_text
    enc_full   = tokenizer(full,        return_tensors='pt', truncation=True,
                           max_length=max_length).to(DEVICE)
    enc_prompt = tokenizer(prompt_text, return_tensors='pt', truncation=True,
                           max_length=max_length).to(DEVICE)
    n_prompt = enc_prompt['input_ids'].shape[1]

    with torch.no_grad():
        logits = model(**enc_full).logits  # (1, seq, vocab)

    # Shift so logits[t] predicts token[t+1]
    shift_logits = logits[:, :-1, :]           # (1, seq-1, vocab)
    shift_labels = enc_full['input_ids'][:, 1:] # (1, seq-1)

    log_probs = F.log_softmax(shift_logits, dim=-1)  # (1, seq-1, vocab)
    token_lp  = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)  # (1, seq-1)

    # Only sum over response tokens
    response_lp = token_lp[:, n_prompt - 1:]  # response starts at index n_prompt-1 after shift
    return response_lp.sum()

print('Log-prob helper defined.')

In [ ]:
# ---------------------------------------------------------------------------
# 8. GRPO training loop — 40 steps, G=8 rollouts per problem
# ---------------------------------------------------------------------------
G          = 8       # rollouts per prompt
CLIP_EPS   = 0.2
LR_GRPO    = 1e-5
N_STEPS    = 40

grpo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
grpo_model.load_state_dict(sft_model.state_dict())  # initialise from SFT

ref_model = copy.deepcopy(grpo_model)
for p in ref_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.Adam(grpo_model.parameters(), lr=LR_GRPO)

step_accuracies = []
step_entropies  = []

for step in range(N_STEPS):
    problem = random.choice(grpo_train)
    prompt  = f"Q: {problem['question']}\nA: Let me solve step by step.\n"
    gt      = problem['answer']
    enc_p   = tokenizer(prompt, return_tensors='pt').to(DEVICE)

    # --- Generate G rollouts ---
    grpo_model.eval()
    responses = []
    rewards   = []
    with torch.no_grad():
        for _ in range(G):
            out = grpo_model.generate(
                **enc_p, max_new_tokens=60, do_sample=True,
                temperature=0.9, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
            response_ids = out[0][enc_p['input_ids'].shape[1]:]
            resp_text = tokenizer.decode(response_ids, skip_special_tokens=True)
            r = verify_answer(resp_text, gt)
            responses.append(resp_text)
            rewards.append(r)

    rewards_t  = torch.tensor(rewards, dtype=torch.float32)
    mean_r     = rewards_t.mean()
    std_r      = rewards_t.std() + 1e-8
    advantages = (rewards_t - mean_r) / std_r  # z-score normalisation

    step_accuracies.append(mean_r.item())

    # --- Policy gradient update ---
    grpo_model.train()
    total_loss = torch.tensor(0.0, device=DEVICE)
    entropy_sum = 0.0

    for resp_text, adv in zip(responses, advantages):
        if resp_text.strip() == '':
            continue
        full = prompt + resp_text
        enc_f = tokenizer(full, return_tensors='pt', truncation=True,
                          max_length=128).to(DEVICE)
        enc_pr = tokenizer(prompt, return_tensors='pt', truncation=True,
                           max_length=128).to(DEVICE)
        n_prompt_tok = enc_pr['input_ids'].shape[1]

        # Current policy log-probs
        logits = grpo_model(**enc_f).logits
        shift_logits = logits[:, :-1, :]
        shift_labels = enc_f['input_ids'][:, 1:]
        log_probs_all = F.log_softmax(shift_logits, dim=-1)
        token_lp = log_probs_all.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
        resp_lp_new = token_lp[:, n_prompt_tok - 1:].sum()

        # Reference log-probs (no grad)
        with torch.no_grad():
            logits_ref = ref_model(**enc_f).logits
            lp_ref_all = F.log_softmax(logits_ref[:, :-1, :], dim=-1)
            token_lp_ref = lp_ref_all.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
            resp_lp_ref = token_lp_ref[:, n_prompt_tok - 1:].sum()

        ratio = torch.exp(resp_lp_new - resp_lp_ref)
        clipped = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS)
        adv_t = adv.to(DEVICE)
        loss  = -torch.min(ratio * adv_t, clipped * adv_t)
        total_loss = total_loss + loss

        # Entropy (proxy) — mean entropy of response token distribution
        probs = torch.exp(log_probs_all[:, n_prompt_tok - 1:, :])
        ent   = -(probs * log_probs_all[:, n_prompt_tok - 1:, :]).sum(-1).mean()
        entropy_sum += ent.item()

    step_entropies.append(entropy_sum / G)

    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(grpo_model.parameters(), 1.0)
    optimizer.step()

    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}/{N_STEPS} | accuracy={mean_r.item():.2f} | '
              f'entropy={entropy_sum/G:.3f} | loss={total_loss.item():.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# 9. Plot accuracy and entropy over GRPO training steps
# ---------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))

# Smooth with a simple moving average
def smooth(vals, w=5):
    return [np.mean(vals[max(0,i-w):i+1]) for i in range(len(vals))]

ax1.plot(smooth(step_accuracies), color='steelblue', label='Accuracy (smoothed)')
ax1.scatter(range(N_STEPS), step_accuracies, alpha=0.3, s=10, color='steelblue')
ax1.set_xlabel('GRPO Step'); ax1.set_ylabel('Fraction Correct')
ax1.set_title('Verifier Accuracy Over GRPO Training')
ax1.legend()

ax2.plot(smooth(step_entropies), color='tomato', label='Entropy (smoothed)')
ax2.scatter(range(N_STEPS), step_entropies, alpha=0.3, s=10, color='tomato')
ax2.set_xlabel('GRPO Step'); ax2.set_ylabel('Entropy')
ax2.set_title('Policy Entropy Over GRPO Training')
ax2.legend()

plt.tight_layout(); plt.show()

---
## Phase 3 — Test-Time Scaling

Test-time compute lets us trade inference FLOPs for accuracy **without retraining**. Two strategies:

1. **Best-of-N (BoN):** generate N candidates, pick the one the verifier accepts (or the one with the highest reward).
2. **Self-consistency (majority vote):** generate N candidates, extract each predicted answer, return the most frequent one.

In [ ]:
# ---------------------------------------------------------------------------
# 10. Best-of-N evaluation on 5 test problems
# ---------------------------------------------------------------------------
def best_of_n(model, tokenizer, question, ground_truth, n):
    """
    Generate n samples, return 1.0 if ANY sample is correct (oracle BoN),
    and also return all responses for majority-vote analysis.
    """
    prompt = f"Q: {question}\nA: Let me solve step by step.\n"
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    responses = []
    model.eval()
    with torch.no_grad():
        for _ in range(n):
            out = model.generate(
                **enc, max_new_tokens=60, do_sample=True,
                temperature=0.9, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
            resp = tokenizer.decode(out[0][enc['input_ids'].shape[1]:],
                                    skip_special_tokens=True)
            responses.append(resp)
    correct_any = any(verify_answer(r, ground_truth) == 1.0 for r in responses)
    return float(correct_any), responses

N_VALUES = [1, 4, 8, 16]
bon_accuracies = {n: [] for n in N_VALUES}  # per N, list over test problems

for prob in grpo_test:
    # Pre-generate 16 samples once to be consistent across N values
    _, all_responses = best_of_n(grpo_model, tokenizer,
                                  prob['question'], prob['answer'], 16)
    for n in N_VALUES:
        subset = all_responses[:n]
        correct = any(verify_answer(r, prob['answer']) == 1.0 for r in subset)
        bon_accuracies[n].append(float(correct))

print('Best-of-N accuracy on 5 test problems:')
for n in N_VALUES:
    acc = np.mean(bon_accuracies[n])
    print(f'  N={n:2d}: {acc:.2f}')

In [ ]:
# ---------------------------------------------------------------------------
# 11. Self-consistency (majority vote) on N=8 samples
# ---------------------------------------------------------------------------
def extract_answer(response):
    m = re.search(r'Answer:\s*(-?\d+)', response)
    return int(m.group(1)) if m else None

def majority_vote(responses, ground_truth):
    answers = [extract_answer(r) for r in responses if extract_answer(r) is not None]
    if not answers:
        return 0.0
    vote, _ = Counter(answers).most_common(1)[0]
    return 1.0 if vote == ground_truth else 0.0

mv_correct = []
for prob in grpo_test:
    _, responses_16 = best_of_n(grpo_model, tokenizer,
                                  prob['question'], prob['answer'], 8)
    mv_correct.append(majority_vote(responses_16, prob['answer']))

print(f'Self-consistency (majority vote, N=8) accuracy: {np.mean(mv_correct):.2f}')

In [ ]:
# ---------------------------------------------------------------------------
# 12. Plot accuracy vs N
# ---------------------------------------------------------------------------
bon_means = [np.mean(bon_accuracies[n]) for n in N_VALUES]

plt.figure(figsize=(6, 3.5))
plt.plot(N_VALUES, bon_means, marker='o', color='steelblue', label='Best-of-N')
plt.axhline(np.mean(mv_correct), color='tomato', linestyle='--', label='Majority vote (N=8)')
plt.xlabel('N (number of samples)'); plt.ylabel('Accuracy')
plt.title('Test-Time Scaling: Accuracy vs N')
plt.xticks(N_VALUES); plt.legend()
plt.tight_layout(); plt.show()

print(f'Best-of-1  accuracy : {bon_means[0]:.2f}')
print(f'Best-of-16 accuracy : {bon_means[-1]:.2f}')
print(f'Gain from scaling   : +{(bon_means[-1]-bon_means[0])*100:.0f} pp')

---
## Ablation Study

We measure accuracy across four checkpoints on 20 problems to isolate the contribution of each phase.

In [ ]:
# ---------------------------------------------------------------------------
# 13. Ablation: baseline → SFT → GRPO → Best-of-8 on 20 problems
# ---------------------------------------------------------------------------
base_model_abl = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

def evaluate_accuracy(model, problems, n_samples=1):
    correct = 0
    for prob in problems:
        prompt = f"Q: {prob['question']}\nA: Let me solve step by step.\n"
        enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
        model.eval()
        with torch.no_grad():
            candidates = []
            for _ in range(n_samples):
                out = model.generate(
                    **enc, max_new_tokens=60, do_sample=(n_samples > 1),
                    temperature=0.9 if n_samples > 1 else 1.0,
                    top_p=0.95, pad_token_id=tokenizer.eos_token_id,
                )
                resp = tokenizer.decode(out[0][enc['input_ids'].shape[1]:],
                                        skip_special_tokens=True)
                candidates.append(resp)
        if any(verify_answer(r, prob['answer']) == 1.0 for r in candidates):
            correct += 1
    return correct / len(problems)

ALL_PROBLEMS = ARITH_PROBLEMS  # 20 problems

print('Running ablation (this may take a minute)...')
acc_baseline = evaluate_accuracy(base_model_abl, ALL_PROBLEMS, n_samples=1)
acc_sft      = evaluate_accuracy(sft_model,      ALL_PROBLEMS, n_samples=1)
acc_grpo     = evaluate_accuracy(grpo_model,     ALL_PROBLEMS, n_samples=1)
acc_bon8     = evaluate_accuracy(grpo_model,     ALL_PROBLEMS, n_samples=8)

print(f'Baseline     : {acc_baseline:.2f}')
print(f'After SFT    : {acc_sft:.2f}')
print(f'After GRPO   : {acc_grpo:.2f}')
print(f'GRPO + BoN-8 : {acc_bon8:.2f}')

In [ ]:
# ---------------------------------------------------------------------------
# 14. Ablation bar chart
# ---------------------------------------------------------------------------
labels  = ['Baseline', 'After SFT', 'After GRPO', 'GRPO + BoN-8']
accs    = [acc_baseline, acc_sft, acc_grpo, acc_bon8]
colours = ['#aec6cf', '#779ecb', '#03468f', '#001f5b']

plt.figure(figsize=(7, 3.5))
bars = plt.bar(labels, accs, color=colours)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2f}', ha='center', fontsize=10)
plt.ylim(0, 1.15); plt.ylabel('Accuracy on 20 Problems')
plt.title('Reasoner Ablation: Training Phase Contributions')
plt.tight_layout(); plt.show()

---
## Error Analysis

Understanding *why* GRPO still fails is as important as knowing *that* it fails. Common failure modes in arithmetic reasoning:

In [ ]:
# ---------------------------------------------------------------------------
# 15. Error analysis — find cases where GRPO still fails (best-of-4)
# ---------------------------------------------------------------------------
print('=== GRPO Error Analysis (best-of-4 on all 20 problems) ===')
failures = []
for prob in ALL_PROBLEMS:
    prompt = f"Q: {prob['question']}\nA: Let me solve step by step.\n"
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    grpo_model.eval()
    with torch.no_grad():
        candidates = []
        for _ in range(4):
            out = grpo_model.generate(
                **enc, max_new_tokens=60, do_sample=True,
                temperature=0.9, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
            resp = tokenizer.decode(out[0][enc['input_ids'].shape[1]:],
                                    skip_special_tokens=True)
            candidates.append(resp)

    if not any(verify_answer(r, prob['answer']) == 1.0 for r in candidates):
        failures.append({'problem': prob, 'samples': candidates})

print(f'Number of failures (best-of-4): {len(failures)}/{len(ALL_PROBLEMS)}')
print()

for i, fail in enumerate(failures[:3]):  # show up to 3 failures
    print(f'--- Failure {i+1} ---')
    print(f'Question    : {fail["problem"]["question"]}')
    print(f'Ground truth: {fail["problem"]["answer"]}')
    print(f'Best sample : {fail["samples"][0][:120]}')
    predicted = extract_answer(fail['samples'][0])
    if predicted is not None:
        print(f'Predicted   : {predicted}')
        diff = abs(predicted - fail['problem']['answer'])
        if diff <= 2:
            print('Failure type: Off-by-small (arithmetic slip in final step)')
        else:
            print('Failure type: Large error (wrong operation or operator precedence)')
    else:
        print('Failure type: No Answer: token generated (format failure)')
    print()

print('Common failure modes in GRPO arithmetic reasoners:')
print('  1. Format failure — model generates text but omits "Answer: N" tag')
print('  2. Operator precedence errors — e.g. treating a-b*c as (a-b)*c')
print('  3. Off-by-one arithmetic — correct steps but final addition slip')
print('  4. Hallucinated intermediate steps — plausible-looking but wrong')

---
## Summary

| Method | Accuracy (20 problems) | Notes |
|--------|------------------------|-------|
| Baseline (pretrained) | ~0.00 | No structured format |
| After CoT SFT | ~0.10 | Learns format, limited generalisation |
| After GRPO | ~0.30 | RL reinforces correct answers |
| GRPO + Best-of-8 | ~0.50 | Test-time compute boosts accuracy further |

*Exact numbers depend on random seed and stochastic sampling.*

**Key takeaways:**
- CoT SFT alone is insufficient for reliable reasoning — it teaches format but not correctness.
- GRPO with a verifier provides a direct training signal that improves correctness without a reward model.
- Test-time scaling (Best-of-N, majority vote) is a powerful complement to RL training.

**Next up:** Chapter 19 extends this to the full *Agent* pipeline with tool-use SFT, trajectory-level RL, and a constitutional safety layer.